In [ ]:
import streamlit as st
from agents import Agent, Runner, WebSearchTool

# ============================================================
# 1. Page config
# ============================================================
st.set_page_config(
    page_title="Life Coach Agent",
    page_icon="🌱",
    layout="centered",
)

In [ ]:
# ============================================================
# 2. Agent setup
# ============================================================
@st.cache_resource
def create_agent():
    return Agent(
        name="Life Coach",
        instructions="""You are a warm, encouraging, and knowledgeable Life Coach.

YOUR ROLE:
- Help users build better habits, stay motivated, and achieve personal growth.
- Provide actionable, evidence-based advice on topics like:
  * Morning routines, sleep habits, waking up early
  * Habit formation and habit stacking
  * Goal setting and time management
  * Mindfulness, stress management, and mental health
  * Fitness, nutrition, and overall well-being
  * Productivity and focus
  * Self-confidence and personal development

BEHAVIOR:
- ALWAYS use the web search tool to find up-to-date, evidence-based tips before answering.
- After searching, summarize the key findings in a friendly, motivating tone.
- Give concrete, step-by-step actionable advice (not vague platitudes).
- Celebrate the user's intention to improve — acknowledge their effort.
- Use a warm, supportive Korean conversational style.
- End responses with an encouraging message or a follow-up question to keep the user engaged.

Respond in Korean.""",
        tools=[WebSearchTool()],
        model="gpt-4o-mini",
    )


agent = create_agent()

In [ ]:
# ============================================================
# 3. Conversation memory
# ============================================================
# input_history: list used for Runner.run_sync(input=...) to maintain multi-turn context
# display_messages: list of {"role", "content"} for Streamlit chat UI rendering
if "input_history" not in st.session_state:
    st.session_state.input_history = []

if "display_messages" not in st.session_state:
    st.session_state.display_messages = []

In [ ]:
# ============================================================
# 4. UI — Header
# ============================================================
st.title("🌱 Life Coach Agent")
st.caption("동기부여, 습관 형성, 자기 개발에 대해 무엇이든 물어보세요! (웹 검색 기반)")

# ============================================================
# 5. UI — Render chat history
# ============================================================
for msg in st.session_state.display_messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# ============================================================
# 6. UI — Chat input + Agent execution
# ============================================================
user_input = st.chat_input("메시지를 입력하세요...")

if user_input:
    # --- Display user message ---
    st.session_state.display_messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)

    # --- Build input for the agent ---
    # Append new user message to the running input history
    agent_input = st.session_state.input_history + [
        {"role": "user", "content": user_input}
    ]

    # --- Run agent with spinner ---
    with st.chat_message("assistant"):
        with st.spinner("검색하고 답변을 준비하고 있어요... 🔍"):
            result = Runner.run_sync(
                agent,
                input=agent_input,
            )

        st.markdown(result.final_output)

    # --- Update session memory ---
    # to_input_list() returns the full conversation in the format the SDK expects
    st.session_state.input_history = result.to_input_list()

    # Save assistant response for display
    st.session_state.display_messages.append(
        {"role": "assistant", "content": result.final_output}
    )